In [1]:
import numpy as np


def fit_mle(X, y):
    """Closed-form OLS: theta = (X^T X)^-1 X^T y"""
    return np.linalg.solve(X.T @ X, X.T @ y)


def fit_map(X, y, lam):
    """
    Closed-form MAP estimate under a Gaussian prior theta ~ N(0, tau^2 I),
    equivalent to Ridge regression with penalty `lam`.
    theta = (X^T X + lam * I)^-1 X^T y
    Bias term (intercept) is not penalized.
    """
    n_features = X.shape[1]
    I = np.eye(n_features)
    I[0, 0] = 0  # don't penalize the intercept column
    return np.linalg.solve(X.T @ X + lam * I, X.T @ y)


In [2]:
from sklearn.linear_model import Ridge
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

data = load_diabetes()
X_raw, y = data.data, data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X = np.column_stack([np.ones(len(X_scaled)), X_scaled])  # add intercept

lambdas = [0.0, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]

print(f"{'lambda':>8} | {'max abs diff vs sklearn Ridge':>30}")
print("-" * 45)
max_diff_overall = 0.0
for lam in lambdas:
    theta_map = fit_map(X, y, lam)
    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_scaled, y)
    sklearn_coefs = np.concatenate([[ridge.intercept_], ridge.coef_])
    diff = np.max(np.abs(theta_map - sklearn_coefs))
    max_diff_overall = max(max_diff_overall, diff)
    print(f"{lam:8.1f} | {diff:30.10f}")

print(f"\nOverall max abs diff across full regularization path: {max_diff_overall:.10f}")
print("Matches sklearn Ridge to 6 decimal places:", max_diff_overall < 1e-6)

theta_mle = fit_mle(X, y)
theta_map_0 = fit_map(X, y, 0.0)
print(f"\nMLE vs MAP(lambda=0) max abs diff: {np.max(np.abs(theta_mle - theta_map_0)):.10f}")


  lambda |  max abs diff vs sklearn Ridge
---------------------------------------------
     0.0 |                   0.0000000000
     0.1 |                   0.0000000000
     1.0 |                   0.0000000000
    10.0 |                   0.0000000000
    50.0 |                   0.0000000000
   100.0 |                   0.0000000000
   500.0 |                   0.0000000000

Overall max abs diff across full regularization path: 0.0000000000
Matches sklearn Ridge to 6 decimal places: True

MLE vs MAP(lambda=0) max abs diff: 0.0000000000


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

scaler = StandardScaler().fit(X_train_raw)
X_train = np.column_stack([np.ones(len(X_train_raw)), scaler.transform(X_train_raw)])
X_test = np.column_stack([np.ones(len(X_test_raw)), scaler.transform(X_test_raw)])

# Estimate noise variance sigma^2 from MLE residuals
theta_mle = fit_mle(X_train, y_train)
resid = y_train - X_train @ theta_mle
n, p = X_train.shape
sigma2 = np.sum(resid ** 2) / (n - p)

# Naive prior
tau_naive = 1.0
lambda_naive = sigma2 / (tau_naive ** 2)
theta_map_naive = fit_map(X_train, y_train, lambda_naive)
pred_naive = X_test @ theta_map_naive
mse_naive = mean_squared_error(y_test, pred_naive)
r2_naive = r2_score(y_test, pred_naive)

# Grounded prior
tau = y_train.std()
lambda_bayesian = sigma2 / (tau ** 2)
theta_map = fit_map(X_train, y_train, lambda_bayesian)
pred_bayesian = X_test @ theta_map
mse_bayesian = mean_squared_error(y_test, pred_bayesian)
r2_bayesian = r2_score(y_test, pred_bayesian)

# sklearn RidgeCV (cross-validated grid search) for comparison
alphas = np.logspace(-2, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, fit_intercept=True)
ridge_cv.fit(scaler.transform(X_train_raw), y_train)
pred_cv = ridge_cv.predict(scaler.transform(X_test_raw))
mse_cv = mean_squared_error(y_test, pred_cv)
r2_cv = r2_score(y_test, pred_cv)

# Plain MLE for reference
pred_mle = X_test @ theta_mle
mse_mle = mean_squared_error(y_test, pred_mle)
r2_mle = r2_score(y_test, pred_mle)

print(f"Estimated noise variance (sigma^2) from MLE residuals: {sigma2:.2f}")
print(f"Naive prior:    tau=1.0             -> lambda={lambda_naive:.2f}")
print(f"Grounded prior: tau=std(y)={tau:.2f}  -> lambda={lambda_bayesian:.2f}")
print(f"RidgeCV's cross-validated best alpha:                    {ridge_cv.alpha_:.2f}")
print()
print(f"{'Method':<30}{'Test MSE':>12}{'Test R2':>10}")
print("-" * 52)
print(f"{'MLE (lambda=0)':<30}{mse_mle:12.2f}{r2_mle:10.4f}")
print(f"{'MAP, naive tau=1':<30}{mse_naive:12.2f}{r2_naive:10.4f}")
print(f"{'MAP, grounded tau=std(y)':<30}{mse_bayesian:12.2f}{r2_bayesian:10.4f}")
print(f"{'Ridge, CV-searched alpha':<30}{mse_cv:12.2f}{r2_cv:10.4f}")


Estimated noise variance (sigma^2) from MLE residuals: 2960.81
Naive prior:    tau=1.0             -> lambda=2960.81
Grounded prior: tau=std(y)=77.95  -> lambda=0.49
RidgeCV's cross-validated best alpha:                    1.15

Method                            Test MSE   Test R2
----------------------------------------------------
MLE (lambda=0)                     2900.19    0.4526
MAP, naive tau=1                   4180.03    0.2110
MAP, grounded tau=std(y)           2895.44    0.4535
Ridge, CV-searched alpha           2891.21    0.4543
